# 06.3 — Multi-Head Attention

**Problem it solves:** Single-head attention computes ONE similarity function. But language has multiple types of relationships simultaneously — syntactic, semantic, positional. Multi-head runs H parallel attention functions, each learning different patterns.

**Architecture:** Project Q, K, V into H smaller dimensions, run attention H times in parallel, concatenate, project back.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)
    attn = torch.nan_to_num(attn, nan=0.0)
    return torch.matmul(attn, V), attn

class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention from 'Attention Is All You Need'.
    
    MHA(Q,K,V) = Concat(head_1,...,head_h) W_O
    head_i = Attention(Q W_i^Q, K W_i^K, V W_i^V)
    
    Implementation: pack all heads into batch dimension for efficiency.
    """
    
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # dimension per head
        
        # 4 linear projections: Q, K, V, and output
        # Note: we could use 3 separate W_Q, W_K, W_V matrices,
        # but one large projection is equivalent and more efficient
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
    
    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """
        Split the last dimension into (n_heads, d_k).
        [batch, seq, d_model] -> [batch, n_heads, seq, d_k]
        """
        batch, seq, _ = x.shape
        x = x.view(batch, seq, self.n_heads, self.d_k)
        return x.transpose(1, 2)  # [batch, n_heads, seq, d_k]
    
    def forward(self, Q, K, V, mask=None):
        """
        Q, K, V: [batch, seq, d_model]
        Returns: [batch, seq_q, d_model]
        """
        batch = Q.size(0)
        
        # 1. Project Q, K, V
        Q = self.split_heads(self.W_Q(Q))  # [batch, heads, seq, d_k]
        K = self.split_heads(self.W_K(K))
        V = self.split_heads(self.W_V(V))
        
        # 2. Compute attention (all heads in parallel)
        attended, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        # attended: [batch, heads, seq, d_k]
        
        # 3. Concatenate heads
        # [batch, heads, seq, d_k] -> [batch, seq, heads*d_k] = [batch, seq, d_model]
        attended = attended.transpose(1, 2).contiguous()
        attended = attended.view(batch, -1, self.d_model)
        
        # 4. Final linear projection
        output = self.W_O(attended)
        
        return output, attn_weights

# Test
d_model, n_heads, seq_len, batch = 64, 8, 10, 2
mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)

x = torch.randn(batch, seq_len, d_model)
out, attn = mha(x, x, x)  # self-attention: Q=K=V=x

print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Attention weights: {attn.shape}  (batch, heads, seq_q, seq_k)")
print(f"d_k per head: {d_model // n_heads}")
print(f"Parameters: {sum(p.numel() for p in mha.parameters()):,}")

In [ ]:
# What do different heads learn?
# Visualization with a trained model would show:
# - Some heads attend to adjacent tokens (local syntax)
# - Some heads attend to specific token types (e.g., always attend to [CLS])
# - Some heads capture long-range dependencies

# Simulate what different heads might attend to
n_heads = 4
seq_len = 6

simulated_heads = {
    0: 'adjacent tokens (local)',
    1: 'subject-verb agreement (syntactic)',
    2: 'co-reference (semantic)',
    3: 'positional (attends to beginning)',
}

print("Each attention head specializes in different relationship types:")
for head_id, description in simulated_heads.items():
    print(f"  Head {head_id}: {description}")

print()
print("Evidence from Clark et al. (2019) 'What Does BERT Look At?':")
print("  - Head 8-10 in BERT layer 2: direct objects of verbs")
print("  - Head 8-11 in BERT layer 5: nominal subjects")
print("  - Head 7-6 in BERT: [SEP] token (delimiter attention)")
print()
print("But: heads are not cleanly interpretable — they're jointly trained.")
print("A head that 'attends to the object' doesn't understand grammar; it finds pattern.") 

# Parameter count: MHA is parameter-efficient
for n_heads in [1, 4, 8, 16]:
    d_model = 512
    n_params = 4 * d_model * d_model  # W_Q, W_K, W_V, W_O
    d_k = d_model // n_heads
    print(f"  {n_heads:2d} heads: d_k={d_k:3d}  params={n_params:,} (same for all!)")
print()
print("MHA has the SAME parameter count regardless of n_heads!")
print("More heads = smaller d_k per head = finer-grained attention")

## Summary

**Multi-head attention key ideas:**
1. Run attention H times in parallel with different projections
2. Each head can specialize in different relationship types
3. Parameter count is independent of number of heads
4. Concatenate and project at the end

**The `split_heads` / `contiguous` / `view` pattern** is the most confusing part of the implementation. It's just tensor reshaping to pack all heads into the batch dimension for efficient parallel computation.

---
**Next:** 06.4 — Positional Encoding